# 🚀 Principal & Staff Level Fintech Technical Interview Mastery (Dual-Engine: Pandas & SQL)

Welcome to the **Staff / Principal Level Fintech Interview Challenge**.

All scenarios in this notebook are strictly **Advanced & Staff Level**, modeled after high-stakes quantitative analytics, risk engineering, and fraud infrastructure interviews at tier-1 institutions (**Two Sigma, Jane Street, Stripe, Brex, Block, Goldman Sachs, American Express**).

---

### ⚡ Rules of the Challenge:
- **Minimal Stakeholder Briefs:** You are given only the high-level business requirement from executive risk and quantitative trading leadership. No hints or intermediate steps are provided.
- **Dual-Engine Execution:** Solve each case in **both Pandas and pure SQL (`%%sql`)**.
- **Advanced Techniques Tested:** Multi-window time-series (`.rolling()`, `.ewm()`, `.shift()`), island-and-gap streak tracking (`(cond != cond.shift()).cumsum()`), Markov transition matrices (`pd.crosstab(normalize='index')`), multi-format parsing (`.xml`, `.dat`, `.psv`, `.tsv`, `.jsonl`, `.yaml`, `.json`), 3-way bridge joins, and vectorized underwriting engines (`np.select()`).

---
### 📁 Multi-Format Enterprise Datasets in `./data/`:
`raw_transactions.csv`, `customers.csv`, `merchants.csv`, `disputes.csv`, `fx_rates_daily.tsv`, `device_telemetry.jsonl`, `api_event_logs.json`, `kyc_audit_records.psv`, `credit_bureau_scores.xml`, `ach_clearing_settlement.dat`, `aml_sanctions_watchlist.yaml`

In [ ]:
# Setup environment & in-memory SQLite relational engine with Native %%sql Execution
import sqlite3
import pandas as pd
import numpy as np
import json
import yaml
import xml.etree.ElementTree as ET
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

# 1. Initialize persistent in-memory SQLite connection
conn = sqlite3.connect(':memory:')

# 2. Universal Table Loader (supports CSV, TSV, and DataFrames)
def load_table(name, source, sep=','):
    if isinstance(source, pd.DataFrame):
        source.to_sql(name, conn, index=False, if_exists='replace')
        print(f"✅ Loaded DataFrame into table: '{name}' ({len(source)} rows)")
    elif isinstance(source, str) and os.path.exists(source):
        df = pd.read_csv(source, sep=sep)
        df.to_sql(name, conn, index=False, if_exists='replace')
        print(f"✅ Loaded '{source}' into table: '{name}' ({len(df)} rows)")

# 3. Pre-load Standard Enterprise Tables
base_dir = 'data' if os.path.exists('data') else '../data'

load_table('transactions', os.path.join(base_dir, 'raw_transactions.csv'))
load_table('customers', os.path.join(base_dir, 'customers.csv'))
load_table('merchants', os.path.join(base_dir, 'merchants.csv'))
load_table('disputes', os.path.join(base_dir, 'disputes.csv'))
load_table('fx_rates', os.path.join(base_dir, 'fx_rates_daily.tsv'), sep='\t')

# 4. Raw SQL Execution Engine
def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# 5. Register %%sql Magic & Auto-Transformer
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("\n🚀 SQL Studio Environment Active! You can now write %%sql directly in code cells.")
print("📁 Available Enterprise Files:", sorted(os.listdir(base_dir)))

---
## 🏢 Case 1: High-Frequency Cross-Border Liquidity Velocity & EWMA Volatility Decay

### 📌 Scenario
> **Stakeholder:** Head of Quantitative Treasury & Liquidity Risk  
> *"Ingest `raw_transactions.csv` and daily market forex rates from `fx_rates_daily.tsv` (Tab-Separated). Convert all completed transactions to USD spot value. For each geographic region, construct a continuous daily time series computing:
> 1. A 7-day rolling total settlement volume.
> 2. A 14-day Exponentially Weighted Moving Average (EWMA, span=14) of daily transaction volume to measure momentum.
> 3. Day-over-Day volume acceleration: `(Daily Volume - 7D Rolling Avg) / 7D Rolling Avg`.
> Rank the top 2 single-day liquidity surge anomalies per region based on acceleration."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 1:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 1:

---
## 🏢 Case 2: Streaming Device Telemetry, Inter-Session Velocity & Tor Exit Contagion

### 📌 Scenario
> **Stakeholder:** Principal Security Data Scientist & Fraud Defense Lead  
> *"Parse streaming telemetry from `device_telemetry.jsonl` (JSON-Lines). For each customer, compute the inter-session latency delta between consecutive device events. Isolate high-frequency automated bot sessions where `network_type == 'Tor-Proxy'` OR `is_rooted_jailbroken == True` with an inter-session latency delta <= 60ms. Join with `customers.csv` to calculate the total compromised account balance exposure, grouped by customer `risk_tier` and `has_crypto_wallet` status, identifying high-risk crypto asset leakage."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 2:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 2:

---
## 🏢 Case 3: XML Credit Bureau Pulls & Multi-Dimensional Underwriting Transition Matrix

### 📌 Scenario
> **Stakeholder:** Chief Credit Risk Officer (CCRO)  
> *"Extract multi-bureau inquiry records from `credit_bureau_scores.xml`. Segment individual bureau FICO scores into standardized Underwriting Bands:
> - `Deep Subprime (<580)`
> - `Near-Prime [580, 669]`
> - `Prime [670, 739]`
> - `Super-Prime [740, 850]`
> Merge with `customers.csv`. Build a normalized cross-tabulation transition matrix comparing internal customer `risk_tier` against bureau score bands, showing both absolute customer counts and row-normalized percentage distributions, while computing total balance exposure for Subprime accounts with `hard_inquiries_12m >= 3`."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 3:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 3:

---
## 🏢 Case 4: Acquiring Interchange Margin Arbitrage & Merchant Net Drain Portfolio

### 📌 Scenario
> **Stakeholder:** Head of Acquiring Economics & Payment Partnerships  
> *"Reconcile transaction processing revenues across `merchants.csv`, `raw_transactions.csv`, and `disputes.csv`:
> 1. Gross Processing Fee = `SUM(transaction_amount * interchange_fee_pct)`.
> 2. Operational Dispute Drain = `SUM(disputed_amount + chargeback_fee_usd)`.
> 3. Net Operating Margin = Gross Fee - Dispute Drain.
> Group by merchant `category` and isolate merchants running at a net negative operating margin (`Net Margin < 0`). Output the top 10 most unprofitable merchant accounts with their category, gross fee, dispute drain, and net loss."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 4:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 4:

---
## 🏢 Case 5: Fixed-Width NACHA Clearing Returns & Systemic Routing Corridor Risk

### 📌 Scenario
> **Stakeholder:** Director of NACHA Settlement & Clearing Compliance  
> *"Ingest the fixed-width NACHA clearing ledger `ach_clearing_settlement.dat` using column width specifications. For every bank routing number (`ROUTING_NUM`), compute the cumulative running sum of total batch settlement volume vs unauthorized returns (`RETURNED_NSF`, `RETURNED_ACT_CLOSED`, `SUSPENDED_AML`). Identify routing corridors whose cumulative unauthorized return rate exceeds 1.5% of total volume. Merge with `customers.csv` to calculate total systemic customer account balance exposure tied to breached routing corridors."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 5:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 5:

---
## 🏢 Case 6: Pipe-Separated PSV Biometric Anomaly & Geographic Spoofing Concentration

### 📌 Scenario
> **Stakeholder:** Head of Identity Intelligence & Biometric Risk  
> *"Parse regulatory verification logs from `kyc_audit_records.psv` (pipe-delimited, handling `#` header comments). Identify synthetic biometric spoofing attempts defined as: `confidence_score < 0.65` AND `facial_match_pct < 55.0` OR `screening_flags` containing `'TAMPERED_DOC_SUSPECTED'`. Cross-reference with `customers.csv` to aggregate biometric anomalies across US states (`state_province`), calculating total verification attempts, spoofing counts, and the biometric spoofing rate per state, ranked by highest risk."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 6:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 6:

---
## 🏢 Case 7: Island-and-Gap MFA Streak Anomaly & Webhook ATO Exposure

### 📌 Scenario
> **Stakeholder:** Staff Security Engineer & Incident Response Lead  
> *"Scan real-time security logs from `api_event_logs.json`. Using consecutive island-and-gap streak tracking on customer event sequences, identify accounts undergoing active credential stuffing attacks defined as: 3 or more consecutive failed MFA challenges (`(mfa_prompted == True) & (mfa_passed == False)`) OR a single event risk score >= 0.90. Merge with `customers.csv` to generate an executive report detailing total attacked accounts, average attack risk score, and total balance exposure grouped by `account_tier`."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 7:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 7:

---
## 🏢 Case 8: Hierarchical YAML Sanctions Graph Contagion & Capital Freeze Engine

### 📌 Scenario
> **Stakeholder:** Chief Compliance Officer (CCO) & FinCEN Liaison  
> *"Ingest OFAC regulatory sanctions from `aml_sanctions_watchlist.yaml`. Scan `customers.csv`, `api_event_logs.json`, and `raw_transactions.csv`. Build a comprehensive multi-vector asset freeze docket flagging any customer meeting ANY of the following 3 criteria:
> 1. Country of registration is an OFAC sanctioned nation.
> 2. Login IP address originates from a sanctioned subnet prefix.
> 3. Completed transaction volume with an associated `is_pep == 1` flag > $50,000.
> Generate an executive FinCEN docket detailing flagged customers, frozen capital, and violation classification."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 8:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 8: